In [4]:
from functions.utils.cloudstorage import GoogleCloudStorage
from functions.utils.bigquery import DataQuery

In [30]:
import json
import numpy as np
import io

from datetime import datetime, timedelta, timezone
from google.cloud import storage

from functions.utils.bigquery import DataQuery

class GoogleCloudStorage:
    '''
    initial instance
    example 
    cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
    '''
    def __init__(self,bucket_name:str):
        self.bucket_name   = bucket_name
        self.client        = storage.Client()
        self.bucket_exists = False
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            self.bucket_exists = True
            print(f"Bucket exists : {bucket_name}")
        except:
            self.bucket = None
            print(f"Bucket NOT exists : {bucket_name}")
        self.hyde_names      = ["hyde_text01.txt","hyde_text02.txt","hyde_text03.txt","hyde_text04.txt","hyde_text05.txt"]
        self.embedding_names = ["embedding01.npy","embedding02.npy","embedding03.npy","embedding04.npy","embedding05.npy"]

    def blob_exists(self, blob_path:str) -> bool:
        '''
        check if object exists
        Ex. blob_exists("stu_p001/hyde/hyde_text01.txt")
        Rt. True/False
        '''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Read file from GCS ---------- ###
    def read_json(self,blob_path:str) -> dict:
        '''
        read json file from gcs
        Ex. read_json("stu_p001/metadata/metadata.json")
        Rt. {}
        '''
        blob = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())
    def read_text(self,blob_path:str) -> str:
        '''
        read text file from gcs
        '''
        blob = self.bucket.blob(blob_path)
        return blob.download_as_text()
    def read_npy(self,blob_path):
        '''
        read .npy (embedding vector) file
        '''
        blob   = self.bucket.blob(blob_path)
        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
    
    ### ---------- Retriev any student data --------- ###
    def _prefix_exists(self,prefix:str) -> bool:
        '''
        - Did any blobs start with this prefix?
        - check are there items on that blob?
        '''
        if not prefix.endswith("/"):
            prefix = prefix + "/"
        blobs = list(self.bucket.list_blobs(prefix=prefix, max_results=1))
        return len(blobs) > 0 
    def _build_metadata_from_biggquery(self,student_id:str) -> dict:
        '''
        if there are no folder then Query data from BigQuery (student,interaction,feed)
        '''
        dq  = DataQuery()
        dqs = dq.get_students([student_id])
        if dqs.empty:
            return {}
        row = dqs.iloc[0].to_dict()
        return {
            "student_id"          : row["student_id"],
            "current_status"      : row["current_status"],
            "education_level"     : row["education_level"],
            "education_major"     : row["education_major"],
            "target_roles"        : row["target_roles"],
            "timezone"            : "UTC",                   # TODO : get value from yaml file
            "model_name"          : "gemini-2.5-flash",      # TODO : get value from yaml file
            "max_output_tokens"   : 1024,                    # TODO : get value from yaml file
            "feed_text_max_chars" : 872,                     # TODO : get value from yaml file
            "temperature"         : 0.1,                     # TODO : get value from yaml file
        }
    def retrieve_student_bundle(self, student_id:str) -> dict:
        results = {
            "metadata"  :{},
            "hyde"      :{},
            "embeddings":{},
            "status"    :""
        }
        ### ----------- bucket ---------- ###
        if self.bucket_exists:
            results["status"] += f"{self.bucket_name} /\n"
        else:
            results["status"] += f"{self.bucket_name} x\n"
        ### --------- student --------- ###
        student_prefix = f"{student_id}"
        if self._prefix_exists(student_prefix):
            results["status"] += f"|- {student_id} /\n"
        else:
            results["status"] += f"|- {student_id} x\n"
        ### ----------preparepart---------- ###
        metadata_prefix  = f"{student_id}/metadata"
        metadata_path    = f"{student_id}/metadata/metadata.json"
        embedding_prefix = f"{student_id}/embedding"
        hyde_prefix      = f"{student_id}/hyde"
        ### ----------metatada---------- ###       
        if self._prefix_exists(metadata_prefix): # if we have blob
            results["status"] += "  |- metadata folder /\n"
            if self.blob_exists(metadata_path):
                results["metadata"] =  self.read_json(metadata_path)
                results["status"] += "    |- metadata.json /\n"
            else:
                results["status"] += "    |- metadata.json x\n"
                # accivate query from BigQuery function
                print(f"activate query data from bigquery function ...")
                results["metadata"] = self._build_metadata_from_bigquery(student_id)
        else:
            results["status"] += "  |- metadata folder x\n"
            results["status"] += "    |- metadata.json x\n"
            print(f"activate query data from bigquery function ...")
            dq  = DataQuery()
            dqs = dq.get_students([student_id])
            dqs_dict = dqs.iloc[0].to_dict()
            results["metadata"] = self._build_metadata_from_bigquery(student_id)
        ### ----------hyde---------- ###
        if self._prefix_exists(hyde_prefix):
            results["status"] += "  |- hyde folder /\n"
            for name in self.hyde_names:
                path = f"{hyde_prefix}/{name}"
                if self.blob_exists(path):
                    results["status"] += f"    |- {path} /\n"
                    results["hyde"][name] = self.read_text(path)
                else:
                    results["hyde"][name] = ""
                    results["status"] += f"    |- {path} x\n"
        else:
            results["status"] += "  |- hyde folder x\n"
            for name in self.hyde_names:
                path = f"{hyde_prefix}/{name}"
                results["status"] += f"    |- {path} x\n"
                results["hyde"][name] = ""
                
        ### ----------embedding---------- ###
        print(f"- embedding_prefix -> {embedding_prefix}")
        if self._prefix_exists(embedding_prefix):
            results["status"] += "  |- embedding folder /\n"
            for name in self.embedding_names:
                path = f"{embedding_prefix}/{name}"
                if self.blob_exists(path):
                    results["status"] += f"    |- {path} /\n"
                    results["embeddings"][name] = self.read_npy(path)
                else:
                    results["embeddings"][name] = np.array([])
                    results["status"] += f"    |- {path} x\n"
        else:
            results["status"] += "  |- embedding folder x\n"
            for name in self.embedding_names:
                path = f"{embedding_prefix}/{name}"
                results["status"] += f"    |- {path} x\n"
                results["embeddings"][name] = np.array([])
                
        return results
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
student_id = "stu_p001"
x = cgs.retrieve_student_bundle(student_id)
print()
print(x['status'])

Bucket exists : hyde-datalake
- embedding_prefix -> stu_p001/embedding

hyde-datalake /
|- stu_p001 /
  |- metadata folder /
    |- metadata.json /
  |- hyde folder /
    |- stu_p001/hyde/hyde_text01.txt /
    |- stu_p001/hyde/hyde_text02.txt /
    |- stu_p001/hyde/hyde_text03.txt x
    |- stu_p001/hyde/hyde_text04.txt /
    |- stu_p001/hyde/hyde_text05.txt /
  |- embedding folder /
    |- stu_p001/embedding/embedding01.npy /
    |- stu_p001/embedding/embedding02.npy /
    |- stu_p001/embedding/embedding03.npy /
    |- stu_p001/embedding/embedding04.npy /
    |- stu_p001/embedding/embedding05.npy /



In [24]:
x["metadata"]

{'student_id': 'stu_p001',
 'current_status': 'student3yr',
 'education_level': 'bachelor',
 'education_major': 'วิทยาการคอมพิวเตอร์',
 'target_roles': 'Data Analyst',
 'timezone': 'UTC',
 'model_name': 'gemini-2.5-flash',
 'max_output_tokens': 2048,
 'feed_text_max_chars': 240,
 'temperature': 0.2}

In [28]:
# x["embeddings"]

In [27]:
x["hyde"]

{'hyde_text01.txt': 'แนวทางสร้างพอร์ต Data Analyst โปรเจกต์ Python SQL',
 'hyde_text02.txt': 'เทคนิคเตรียมสัมภาษณ์ฝึกงาน Data Analyst โจทย์ SQL Python',
 'hyde_text03.txt': '',
 'hyde_text04.txt': 'เครื่องมือสร้างแดชบอร์ดข้อมูล Power BI Tableau',
 'hyde_text05.txt': 'แนวโน้มอาชีพ Data Analyst ทักษะที่ตลาดต้องการ'}

<hr>

### case 1 : happy

In [53]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
student_id = "stu_p001"
cgs.retrieve_student_bundle(student_id,'xxx')

Bucket exists  : hyde-datalake


TypeError: Bucket.list_blobs() got multiple values for argument 'max_results'

TypeError: Bucket.list_blobs() got multiple values for argument 'max_results'

<hr>

In [ ]:
### case 2.3 : 
bucket                          /
|- embedding                    x
    |- embedding01.npy          o
    |- embedding02.npy          o
    |- embedding03.npy          o
    |- embedding04.npy          o
    |- embedding05.npy          o
|- metadata                     /
    |- metadata.json            /

In [ ]:
### case 2.4 : 
bucket                          /
|- embedding                    /
    |- embedding01.npy          /
    |- embedding02.npy          /
    |- embedding03.npy          x
    |- embedding04.npy          x
    |- embedding05.npy          x
|- metadata                     /
    |- metadata.json            /

In [ ]:
### case 2.5 : 
bucket                          /
|- embedding                    /
    |- embedding01.npy          x
    |- embedding02.npy          x
    |- embedding03.npy          x
    |- embedding04.npy          x
    |- embedding05.npy          x
|- metadata                     /
    |- metadata.json            /

<hr>

In [ ]:
### case 3 : 
bucket                          x
|- embedding                    o
    |- embedding01.npy          o
    |- embedding02.npy          o
    |- embedding03.npy          o
    |- embedding04.npy          o
    |- embedding05.npy          o
|- metadta                      o
    |- metadata.json            o